In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
import os
print(os.getcwd())

/home/73f3ec34-1f49-46dd-ae6e-0a3e030712ff


In [7]:
import pandas as pd

df = pd.read_csv("fake_job_postings.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (7579, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0.0,1.0,0.0,Other,Internship,NaN,NaN,Marketing,0.0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0.0,1.0,0.0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0.0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0.0,1.0,0.0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0.0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0.0,1.0,1.0,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0.0


In [8]:
# Check missing values

print("Missing values before preprocessing:")
print(df.isnull().sum())

Missing values before preprocessing:
job_id                    0
title                     0
location                144
department             4788
salary_range           6365
company_profile        1663
description               0
requirements           1012
benefits               3012
telecommuting             1
has_company_logo          1
has_questions             1
employment_type        1771
required_experience    3090
required_education     3425
industry               2247
function               2804
fraudulent                1
dtype: int64


In [9]:
# Handle missing values

text_columns = [
    'location',
    'department',
    'salary_range',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'employment_type',
    'required_experience',
    'required_education',
    'industry',
    'function'
]

for column in text_columns:
    df[column] = df[column].fillna('Unknown')

# Fill numeric columns
numeric_columns = [
    'telecommuting',
    'has_company_logo',
    'has_questions'
]

for column in numeric_columns:
    df[column] = df[column].fillna(0)

# Remove rows where target value is missing
df = df.dropna(subset=['fraudulent'])

print("Missing values handled successfully!")
print("Remaining rows:", len(df))

Missing values handled successfully!
Remaining rows: 7578


In [10]:
# Combine important text columns

df['combined_text'] = (
    df['title'] + ' ' +
    df['location'] + ' ' +
    df['company_profile'] + ' ' +
    df['description'] + ' ' +
    df['requirements'] + ' ' +
    df['benefits'] + ' ' +
    df['employment_type'] + ' ' +
    df['required_experience'] + ' ' +
    df['required_education'] + ' ' +
    df['industry'] + ' ' +
    df['function']
)

print("Text features combined successfully!")
print("Total records:", len(df))

Text features combined successfully!
Total records: 7578


In [11]:
# Define input and target

X = df['combined_text']
y = df['fraudulent']

print("Input features:", X.shape)
print("Target values:", y.shape)

Input features: (7578,)
Target values: (7578,)


In [12]:
# Split the dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 6062
Testing samples: 1516


In [13]:
# Convert text into numerical features using TF-IDF

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF conversion completed!")
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)

TF-IDF conversion completed!
Training matrix shape: (6062, 5000)
Testing matrix shape: (1516, 5000)


In [14]:
# Train Logistic Regression model

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_tfidf, y_train)

print("Machine Learning model trained successfully!")

Machine Learning model trained successfully!


In [15]:
# Predict fake or real job postings

y_pred = model.predict(X_test_tfidf)

print("Prediction completed successfully!")
print("First 10 predictions:", y_pred[:10])

Prediction completed successfully!
First 10 predictions: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [16]:
# Calculate model accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Accuracy: 96.44 %


In [17]:
# Classification report

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Real Job', 'Fake Job']
))

Classification Report:
              precision    recall  f1-score   support

    Real Job       0.99      0.97      0.98      1444
    Fake Job       0.59      0.79      0.68        72

    accuracy                           0.96      1516
   macro avg       0.79      0.88      0.83      1516
weighted avg       0.97      0.96      0.97      1516



In [19]:
# Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1405   39]
 [  15   57]]


In [20]:
# Test a new job posting

new_job = """
Work from home job.
Earn Rs.50000 per week.
No experience required.
Pay registration fee of Rs.2000.
Send your bank details and personal information immediately.
"""

new_job_tfidf = vectorizer.transform([new_job])

prediction = model.predict(new_job_tfidf)

if prediction[0] == 1:
    print("Prediction: FAKE JOB POSTING")
else:
    print("Prediction: REAL JOB POSTING")

Prediction: REAL JOB POSTING


In [21]:
# Test another job posting

new_job = """
Software Developer required for a technology company.
The candidate should have knowledge of Python, Java,
SQL and software development. Bachelor's degree in
Computer Science is preferred. Full-time position.
Salary and benefits will be provided according to company policy.
"""

new_job_tfidf = vectorizer.transform([new_job])

prediction = model.predict(new_job_tfidf)

if prediction[0] == 1:
    print("Prediction: FAKE JOB POSTING")
else:
    print("Prediction: REAL JOB POSTING")

Prediction: REAL JOB POSTING


In [22]:
print("==========================================")
print(" FAKE JOB POSTING DETECTION SYSTEM")
print("==========================================")
print("Algorithm : Logistic Regression")
print("Feature Extraction : TF-IDF")
print("Dataset Size :", len(df))
print("Accuracy :", round(accuracy * 100, 2), "%")
print("==========================================")
print("Project completed successfully!")

 FAKE JOB POSTING DETECTION SYSTEM
Algorithm : Logistic Regression
Feature Extraction : TF-IDF
Dataset Size : 7578
Accuracy : 96.44 %
Project completed successfully!
